# Imports

In [2]:
from pyiron_atomistics import Project
import pandas as pd


# Create pyiron project

In [18]:
pr = Project("../cu_zr_mlip_project")
structure = pr.create.structure.bulk("Cu", cubic=True)


# Create potential MACE

In [19]:
compiled_model="CuZr_mace_debug.model-lammps.pt"
MACE_potential = pd.DataFrame({
    "Name": ["MACE_CuZr"],
    # these files will be copied into the job working directory
    "Filename": [[compiled_model]],
    "Model": ["Custom"],
    # IMPORTANT: order must match LAMMPS atom types in structure.inp
    "Species": [["Cu", "Zr"]],
    # These lines become potential.inp
    "Config": [[
        "pair_style mace\n",
        f"pair_coeff * * {compiled_model} Cu Zr\n",
    ]],
})

# Create LAMMPS job

In [20]:
job = pr.create.job.Lammps("mace_md_test", delete_existing_job=True)
job.structure = structure
job.potential = MACE_potential


# MD settings

In [21]:
job.calc_md(temperature=300, n_ionic_steps=1000)


# Run

In [22]:
job.run()

The job mace_md_test was saved and received the ID: 8


# MD integrator

In [10]:
dyn = Langevin(
    atoms,
    timestep=1.0 * units.fs,
    temperature_K=300,
    friction=0.01
)

# Attach trajectory writer

In [12]:
traj_file = "mace_md.traj"

def write_frame():
    write(traj_file, atoms, append=True)

dyn.attach(write_frame, interval=10)

# Run MD

In [13]:
dyn.run(2000)   # steps
